# 17 — Build Target Model Input Table (Ward25)

This notebook creates a clean modelling input table from the K7 Ward25 atlas plus the latest election metrics.

It is deliberately **scope-configurable**:

- Default analytical scope: **North West**
- Optional wider scopes: England-wide, Wales, or individual regions if an LAD→Region lookup is available
- Output is not a final targeting decision; it is a consolidated modelling input for later review/scoring.

Expected main input:

`k7_ward25_atlas_with_latest_election_metrics_v1.csv`

## 17.1 Project paths and configuration

The notebook is designed to run from:

`C:\Users\keena\Documents\Electoral_Tribes\notebooks`

It searches under `data/processed` for the latest K7 Ward25 atlas/election file, so it is tolerant of small folder differences created by earlier notebooks.

In [4]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import json

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

NOTEBOOK_DIR = Path.cwd()

if NOTEBOOK_DIR.name.lower() == "notebooks":
    PROJECT_DIR = NOTEBOOK_DIR.parent
else:
    PROJECT_DIR = NOTEBOOK_DIR

DATA_DIR = PROJECT_DIR / "data"
PROCESSED_DIR = DATA_DIR / "processed"
GEOGRAPHY_DIR = DATA_DIR / "geography"
DICTIONARY_DIR = DATA_DIR / "dictionaries"

MODEL_DIR = PROCESSED_DIR / "target_model_v1"
MODEL_INPUT_DIR = MODEL_DIR / "inputs"
MODEL_OUTPUT_DIR = MODEL_DIR / "outputs"
MODEL_REVIEW_DIR = MODEL_DIR / "review"

for d in [MODEL_DIR, MODEL_INPUT_DIR, MODEL_OUTPUT_DIR, MODEL_REVIEW_DIR, DICTIONARY_DIR]:
    d.mkdir(parents=True, exist_ok=True)

K = 7

DEFAULT_SCOPE = "north_west"
GENERATE_REGION_OUTPUTS = True   # Change to True later when you want region-by-region files.
GENERATE_NATIONAL_OUTPUTS = True   # Keeps an England/Wales-ready modelling input.

print("Project:", PROJECT_DIR)
print("Processed:", PROCESSED_DIR)
print("Model dir:", MODEL_DIR)

Project: c:\Users\keena\Documents\Electoral_Tribes
Processed: c:\Users\keena\Documents\Electoral_Tribes\data\processed
Model dir: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1


## 17.2 Locate the atlas/election file

This should usually be an output from Notebook 15. If several copies exist, the notebook prefers files under `data/processed` and chooses the newest modified copy.

In [5]:
def find_latest_file(root: Path, filename: str) -> Path:
    matches = list(root.rglob(filename))
    if not matches:
        raise FileNotFoundError(f"Could not find {filename} under {root}")
    matches = sorted(matches, key=lambda p: p.stat().st_mtime, reverse=True)
    return matches[0]

ATLAS_FILENAME = "k7_ward25_atlas_with_latest_election_metrics_v1.csv"
ATLAS_PATH = find_latest_file(PROCESSED_DIR, ATLAS_FILENAME)

print("Using atlas/election file:")
print(ATLAS_PATH)

atlas = pd.read_csv(ATLAS_PATH, low_memory=False)

print("Rows:", len(atlas))
print("Columns:", len(atlas.columns))
display(atlas.head())

Using atlas/election file:
c:\Users\keena\Documents\Electoral_Tribes\data\processed\election_results\k7_atlas_join_v1\k7_ward25_atlas_with_latest_election_metrics_v1.csv
Rows: 7572
Columns: 203


,LAD25CD,LAD25NM,WD25CD,WD25NM,population,oa_count,cluster_0_population,cluster_1_population,cluster_2_population,cluster_3_population,cluster_4_population,cluster_5_population,cluster_6_population,cluster_0_share,cluster_1_share,cluster_2_share,cluster_3_share,cluster_4_share,cluster_5_share,cluster_6_share,dominant_cluster,second_cluster,dominant_cluster_share,second_cluster_share,cluster_fragmentation_index,dominant_cluster_name,second_cluster_name,student_transient_youth_share,student_transient_youth_population,rooted_older_homeowners_share,rooted_older_homeowners_population,stable_suburban_professionals_share,stable_suburban_professionals_population,cosmopolitan_young_professional_core_share,cosmopolitan_young_professional_core_population,settled_working_families_skilled_trades_suburbs_share,settled_working_families_skilled_trades_suburbs_population,settled_diverse_urban_communities_share,settled_diverse_urban_communities_population,post_industrial_estates_deprived_working_communities_share,post_industrial_estates_deprived_working_communities_population,geography_level,aggregation_method,source_lookup,total_residents,household_residents,communal_establishment_residents,age_0_14_count,age_15_24_count,age_25_34_count,age_35_49_count,age_50_64_count,age_65_plus_count,partnership_total,never_married_count,married_or_civil_partnership_count,separated_count,divorced_or_dissolved_count,widowed_count,household_composition_total,one_person_household_count,one_person_66_plus_count,single_family_household_count,married_couple_family_count,cohabiting_couple_family_count,lone_parent_family_count,other_household_types_count,country_of_birth_total,uk_born_count,eu_born_count,non_uk_born_count,non_uk_non_eu_born_count,length_residence_total,born_in_uk_count,resident_10_plus_years_count,resident_5_to_10_years_count,resident_2_to_5_years_count,resident_less_2_years_count,resident_less_5_years_count,ethnic_group_total,asian_count,black_count,mixed_count,white_count,white_british_count,white_other_count,other_ethnic_group_count,non_white_count,accommodation_total,detached_count,semi_detached_count,terraced_count,purpose_built_flat_count,converted_flat_count,commercial_building_flat_count,caravan_mobile_temp_count,house_type_count,flat_type_count,tenure_total,owned_count,...,social_rented_count,private_rented_count,lives_rent_free_count,occupation_total,managerial_professional_count,administrative_secretarial_count,skilled_traditional_count,routine_service_elementary_count,economic_activity_total,employed_count,unemployed_count,full_time_student_count,economically_inactive_count,retired_count,looking_after_home_family_count,long_term_sick_disabled_count,qualification_total,no_qualifications_count,level_1_2_count,apprenticeship_count,level_3_count,level_4_plus_count,other_qualifications_count,age_0_14_pct,age_15_24_pct,age_25_34_pct,age_35_49_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,owns_mortgage_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct,is_mixed_ward,is_clear_dominant_ward,is_highly_fragmented,latest_election_WD25NM,latest_election_LAD25CD,latest_election_LAD25NM,latest_election_source_year,latest_election_allocated_electorate,latest_election_allocated_valid_votes,latest_election_allocated_ballots,latest_election_allocated_invalid_votes,latest_election_allocated_top_party_votes,latest_election_allocated_runner_up_party_votes,latest_election_allocated_con_votes,latest_election_allocated_lab_votes,latest_election_allocated_ld_votes,latest_election_allocated

## 17.3 Optional LAD25 → Region lookup

For North West-only work, the notebook can fall back to a built-in North West LAD list.

For England-wide regional outputs, place an ONS LAD-to-region lookup in:

`data/geography`

The preferred fields are:

`LAD25CD`, `LAD25NM`, `RGN25CD`, `RGN25NM`

Likely file names include something like:

`lad25_to_rgn25_eng.csv`

or the full ONS download name.

In [6]:
def standardise_region_lookup_columns(df: pd.DataFrame) -> pd.DataFrame:
    # Keep original names but find columns case-insensitively.
    colmap = {c.lower().strip(): c for c in df.columns}

    def find_col(candidates):
        for cand in candidates:
            if cand.lower() in colmap:
                return colmap[cand.lower()]
        return None

    lad_cd = find_col(["LAD25CD", "lad25cd", "LADCD", "ladcd"])
    lad_nm = find_col(["LAD25NM", "lad25nm", "LADNM", "ladnm"])
    rgn_cd = find_col(["RGN25CD", "rgn25cd", "RGNCD", "rgncd"])
    rgn_nm = find_col(["RGN25NM", "rgn25nm", "RGNNM", "rgnnm"])

    required = [lad_cd, lad_nm, rgn_cd, rgn_nm]
    if any(x is None for x in required):
        return pd.DataFrame()

    out = df[[lad_cd, lad_nm, rgn_cd, rgn_nm]].copy()
    out.columns = ["LAD25CD", "LAD25NM_lookup", "RGN25CD", "RGN25NM"]
    out["LAD25CD"] = out["LAD25CD"].astype(str).str.strip()
    return out.drop_duplicates("LAD25CD")


def find_region_lookup(geo_dir: Path) -> Path | None:
    candidates = []
    for p in geo_dir.glob("*"):
        if p.suffix.lower() not in [".csv", ".xlsx", ".xls"]:
            continue
        name = p.name.lower()
        if ("lad25" in name or "local_authority_district" in name or "local authority district" in name) and ("rgn" in name or "region" in name):
            candidates.append(p)
    if not candidates:
        return None
    return sorted(candidates, key=lambda p: p.stat().st_mtime, reverse=True)[0]


REGION_LOOKUP_PATH = find_region_lookup(GEOGRAPHY_DIR)

if REGION_LOOKUP_PATH is not None:
    print("Region lookup found:", REGION_LOOKUP_PATH)
    if REGION_LOOKUP_PATH.suffix.lower() in [".xlsx", ".xls"]:
        region_raw = pd.read_excel(REGION_LOOKUP_PATH)
    else:
        region_raw = pd.read_csv(REGION_LOOKUP_PATH, low_memory=False)
    region_lookup = standardise_region_lookup_columns(region_raw)
    if len(region_lookup) == 0:
        print("Region lookup was found but required columns were not detected. Falling back to built-in North West list.")
        REGION_LOOKUP_AVAILABLE = False
    else:
        REGION_LOOKUP_AVAILABLE = True
        atlas = atlas.merge(region_lookup[["LAD25CD", "RGN25CD", "RGN25NM"]], on="LAD25CD", how="left", validate="many_to_one")
else:
    print("No LAD25→Region lookup found. Falling back to built-in North West LAD list only.")
    REGION_LOOKUP_AVAILABLE = False
    atlas["RGN25CD"] = np.nan
    atlas["RGN25NM"] = np.nan

Region lookup found: c:\Users\keena\Documents\Electoral_Tribes\data\geography\lad25_to_rgn25_eng.csv


## 17.4 Create scope flags

The default analytical scope is the North West.

If a region lookup is available, region-level outputs can later be generated using `RGN25NM`.

The scope flags make it easy to widen or narrow the model later without rewriting the scoring logic.

In [7]:
NORTH_WEST_LADS = [
    "Cheshire East",
    "Cheshire West and Chester",
    "Halton",
    "Warrington",
    "Cumberland",
    "Westmorland and Furness",
    "Bolton",
    "Bury",
    "Manchester",
    "Oldham",
    "Rochdale",
    "Salford",
    "Stockport",
    "Tameside",
    "Trafford",
    "Wigan",
    "Blackburn with Darwen",
    "Blackpool",
    "Burnley",
    "Chorley",
    "Fylde",
    "Hyndburn",
    "Lancaster",
    "Pendle",
    "Preston",
    "Ribble Valley",
    "Rossendale",
    "South Ribble",
    "West Lancashire",
    "Wyre",
    "Knowsley",
    "Liverpool",
    "Sefton",
    "St. Helens",
    "Wirral",
]

atlas["scope_north_west"] = atlas["LAD25NM"].isin(NORTH_WEST_LADS)

# Basic country inference from LAD code.
atlas["country_inferred"] = np.select(
    [
        atlas["LAD25CD"].astype(str).str.startswith("E"),
        atlas["LAD25CD"].astype(str).str.startswith("W"),
    ],
    ["England", "Wales"],
    default="Unknown"
)

atlas["scope_england"] = atlas["country_inferred"].eq("England")
atlas["scope_wales"] = atlas["country_inferred"].eq("Wales")
atlas["scope_all_available"] = True

# If region lookup is missing, still label North West where known.
atlas["analysis_region"] = atlas["RGN25NM"]
atlas.loc[atlas["scope_north_west"], "analysis_region"] = atlas.loc[atlas["scope_north_west"], "analysis_region"].fillna("North West")
atlas["analysis_region"] = atlas["analysis_region"].fillna(atlas["country_inferred"])

scope_summary = pd.DataFrame({
    "scope": ["north_west", "england", "wales", "all_available"],
    "row_count": [
        atlas["scope_north_west"].sum(),
        atlas["scope_england"].sum(),
        atlas["scope_wales"].sum(),
        atlas["scope_all_available"].sum(),
    ]
})

display(scope_summary)

scope_summary.to_csv(MODEL_REVIEW_DIR / "model_scope_row_counts_v1.csv", index=False)

C:\Users\keena\AppData\Local\Temp\ipykernel_15112\4108437004.py:39: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  atlas["scope_north_west"] = atlas["LAD25NM"].isin(NORTH_WEST_LADS)
C:\Users\keena\AppData\Local\Temp\ipykernel_15112\4108437004.py:42: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  atlas["country_inferred"] = np.select(
C:\Users\keena\AppData\Local\Temp\ipykernel_15112\4108437004.py:51: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor

,scope,row_count
0,north_west,825
1,england,6810
2,wales,762
3,all_available,7572


## 17.5 Select modelling columns

This cell keeps the model input intentionally narrow. The full atlas remains available separately, but the scoring notebook should not depend on hundreds of columns.

In [8]:
def existing(cols):
    return [c for c in cols if c in atlas.columns]

core_cols = [
    "LAD25CD", "LAD25NM", "WD25CD", "WD25NM",
    "RGN25CD", "RGN25NM", "analysis_region", "country_inferred",
    "scope_north_west", "scope_england", "scope_wales", "scope_all_available",
    "population", "oa_count",
    "dominant_cluster", "dominant_cluster_name",
    "second_cluster", "second_cluster_name",
    "dominant_cluster_share", "second_cluster_share",
    "cluster_fragmentation_index",
    "is_mixed_ward", "is_clear_dominant_ward", "is_highly_fragmented",
]

cluster_cols = [
    "cluster_0_share", "cluster_1_share", "cluster_2_share", "cluster_3_share", "cluster_4_share", "cluster_5_share", "cluster_6_share",
    "student_transient_youth_share",
    "rooted_older_homeowners_share",
    "stable_suburban_professionals_share",
    "cosmopolitan_young_professional_core_share",
    "settled_working_families_skilled_trades_suburbs_share",
    "settled_diverse_urban_communities_share",
    "post_industrial_estates_deprived_working_communities_share",
]

demographic_cols = [
    "age_0_14_pct", "age_15_24_pct", "age_25_34_pct", "age_35_49_pct", "age_50_64_pct", "age_65_plus_pct",
    "uk_born_pct", "non_uk_born_pct", "resident_10_plus_years_pct", "resident_less_5_years_pct",
    "white_british_pct", "white_other_pct", "non_white_pct",
    "owned_pct", "owns_outright_pct", "owns_mortgage_pct", "social_rented_pct", "private_rented_pct",
    "house_type_pct", "flat_type_pct",
    "managerial_professional_pct", "skilled_traditional_pct", "routine_service_elementary_pct",
    "employed_pct", "unemployed_pct", "full_time_student_pct", "retired_pct", "long_term_sick_disabled_pct",
    "no_qualifications_pct", "level_1_2_pct", "apprenticeship_pct", "level_4_plus_pct",
    "one_person_household_pct", "married_couple_family_pct", "lone_parent_family_pct",
]

election_cols = [
    "latest_election_WD25NM", "latest_election_LAD25CD", "latest_election_LAD25NM",
    "latest_election_source_year",
    "latest_election_allocated_electorate",
    "latest_election_allocated_valid_votes",
    "latest_election_allocated_ballots",
    "latest_election_allocated_invalid_votes",
    "latest_election_allocated_top_party_votes",
    "latest_election_allocated_runner_up_party_votes",
    "latest_election_allocated_con_votes",
    "latest_election_allocated_lab_votes",
    "latest_election_allocated_ld_votes",
    "latest_election_allocated_green_votes",
    "latest_election_allocated_reform_ukip_brexit_votes",
    "latest_election_allocated_independent_votes",
    "latest_election_allocated_sdp_votes",
    "latest_election_allocated_other_votes",
    "latest_election_contributing_oa_rows",
    "latest_election_contributing_result_areas",
    "latest_election_contributing_source_years",
    "latest_election_con_share",
    "latest_election_lab_share",
    "latest_election_ld_share",
    "latest_election_green_share",
    "latest_election_reform_ukip_brexit_share",
    "latest_election_independent_share",
    "latest_election_sdp_share",
    "latest_election_other_share",
    "latest_election_top_party_bucket",
    "latest_election_runner_up_party_bucket",
    "latest_election_top_party_votes_allocated",
    "latest_election_runner_up_party_votes_allocated",
    "latest_election_margin_votes_allocated",
    "latest_election_margin_pct_allocated",
    "latest_election_party_fragmentation_index",
    "latest_election_effective_number_of_parties",
    "latest_election_aggregation_label",
    "latest_election_latest_layer_note",
]

model_cols = existing(core_cols + cluster_cols + demographic_cols + election_cols)
model_input = atlas[model_cols].copy()

print("Model input columns:", len(model_input.columns))
print("Model input rows:", len(model_input))
display(model_input.head())

Model input columns: 112
Model input rows: 7572


,LAD25CD,LAD25NM,WD25CD,WD25NM,RGN25CD,RGN25NM,analysis_region,country_inferred,scope_north_west,scope_england,scope_wales,scope_all_available,population,oa_count,dominant_cluster,dominant_cluster_name,second_cluster,second_cluster_name,dominant_cluster_share,second_cluster_share,cluster_fragmentation_index,is_mixed_ward,is_clear_dominant_ward,is_highly_fragmented,cluster_0_share,cluster_1_share,cluster_2_share,cluster_3_share,cluster_4_share,cluster_5_share,cluster_6_share,student_transient_youth_share,rooted_older_homeowners_share,stable_suburban_professionals_share,cosmopolitan_young_professional_core_share,settled_working_families_skilled_trades_suburbs_share,settled_diverse_urban_communities_share,post_industrial_estates_deprived_working_communities_share,age_0_14_pct,age_15_24_pct,age_25_34_pct,age_35_49_pct,age_50_64_pct,age_65_plus_pct,uk_born_pct,non_uk_born_pct,resident_10_plus_years_pct,resident_less_5_years_pct,white_british_pct,white_other_pct,non_white_pct,owned_pct,owns_outright_pct,owns_mortgage_pct,social_rented_pct,private_rented_pct,house_type_pct,flat_type_pct,managerial_professional_pct,skilled_traditional_pct,routine_service_elementary_pct,employed_pct,unemployed_pct,full_time_student_pct,retired_pct,long_term_sick_disabled_pct,no_qualifications_pct,level_1_2_pct,apprenticeship_pct,level_4_plus_pct,one_person_household_pct,married_couple_family_pct,lone_parent_family_pct,latest_election_WD25NM,latest_election_LAD25CD,latest_election_LAD25NM,latest_election_source_year,latest_election_allocated_electorate,latest_election_allocated_valid_votes,latest_election_allocated_ballots,latest_election_allocated_invalid_votes,latest_election_allocated_top_party_votes,latest_election_allocated_runner_up_party_votes,latest_election_allocated_con_votes,latest_election_allocated_lab_votes,latest_election_allocated_ld_votes,latest_election_allocated_green_votes,latest_election_allocated_reform_ukip_brexit_votes,latest_election_allocated_independent_votes,latest_election_allocated_sdp_votes,latest_election_allocated_other_votes,latest_election_contributing_oa_rows,latest_election_contributing_result_areas,latest_election_contributing_source_years,latest_election_con_share,latest_election_lab_share,latest_election_ld_share,latest_election_green_share,latest_election_reform_ukip_brexit_share,latest_election_independent_share,latest_election_sdp_share,latest_election_other_share,latest_election_top_party_bucket,latest_election_runner_up_party_bucket,latest_election_top_party_votes_allocated,latest_election_runner_up_party_votes_allocated,latest_election_margin_votes_allocated,latest_election_margin_pct_allocated,latest_election_party_fragmentation_index,latest_election_effective_number_of_parties,latest_election_aggregation_label,latest_election_latest_layer_note
0,E06000001,Hartlepool,E05013038,Burn Valley,E12000001,North East,North East,England,False,True,False,True,7633,26,6.0,Post-Industrial Estates / Deprived Working Com...,2.0,Stable Suburban Professionals,0.458404,0.191406,0.703925,False,False,True,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.043495,0.167169,0.191406,0.0,0.139526,0.0,0.458404,0.174112,0.137037,0.106511,0.179353,0.206996,0.194157,0.948513,0.051487,0.029208,0.014931,0.925465,0.018863,0.052135,0.554228,0.304751,0.249477,0.217807,0.224081,0.843032,0.155774,0.371084,0.207229,0.333219,0.467880,0.046208,0.092256,0.234262,0.084688,0.225000,0.240806,0.062903,0.267419,0.389420,0.251046,0.124626,Burn Valley,E06000001,Hartlepool,2024.0,5818.0,1686.0,0.0,0.0,919.0,339.0,339.0,919.0,0.0,0.0,270.0,158.0,0.0,0.0,26.0,1.0,2024.0,0.201068,0.545077,0.0,0.0,0.160142,0.093713,0.0,0.000000,lab,con,919.0,339.0,580.0,0.344009,0.628035,2.688426,ward25_by_source_year,latest_available_source_year_after_oa21_crosswalk
1,E06000001,Hartlepool,E05013039,De Bruce,E12000001,North East,North East,England,False,True,False,True,8055,25,6.0,Post-Industrial Estates / Deprived Working Com...,4.0,Settled Working Families / Skilled 

## 17.6 Add data-confidence and caveat fields

These fields should be carried through to every later model output.

In [9]:
# Numeric clean-up for key election fields.
for col in [
    "latest_election_source_year",
    "latest_election_allocated_valid_votes",
    "latest_election_allocated_electorate",
    "latest_election_margin_pct_allocated",
]:
    if col in model_input.columns:
        model_input[col] = pd.to_numeric(model_input[col], errors="coerce")

model_input["has_latest_election_layer"] = model_input.get("latest_election_source_year", pd.Series(index=model_input.index)).notna()
model_input["has_valid_vote_data"] = model_input.get("latest_election_allocated_valid_votes", pd.Series(index=model_input.index)).fillna(0).gt(0)
model_input["has_margin_data"] = model_input.get("latest_election_margin_pct_allocated", pd.Series(index=model_input.index)).notna()

model_input["boundary_caveat"] = ""

# Sefton has known 2026 ward change issues in this project.
model_input.loc[model_input["LAD25NM"].eq("Sefton"), "boundary_caveat"] = "Sefton WD25 used for atlas; 2026 ward structure changed. Treat mapping as interim."

# Lancashire county results may include county electoral division influence where latest layer comes from 2025.
# This is a cautious note, not an exclusion rule.
lancashire_lads = [
    "Blackburn with Darwen", "Blackpool", "Burnley", "Chorley", "Fylde",
    "Hyndburn", "Lancaster", "Pendle", "Preston", "Ribble Valley",
    "Rossendale", "South Ribble", "West Lancashire", "Wyre",
]
model_input["county_election_caveat"] = np.where(
    model_input["LAD25NM"].isin(lancashire_lads) & model_input.get("latest_election_source_year", pd.Series(index=model_input.index)).eq(2025),
    "Latest election layer may include 2025 county electoral division allocation; check before fine-grained use.",
    ""
)

model_input["target_model_ready"] = (
    model_input["has_latest_election_layer"]
    & model_input["has_valid_vote_data"]
    & model_input["has_margin_data"]
)

model_input["data_confidence_note"] = np.select(
    [
        ~model_input["has_latest_election_layer"],
        ~model_input["has_valid_vote_data"],
        ~model_input["has_margin_data"],
        model_input["boundary_caveat"].ne(""),
        model_input["county_election_caveat"].ne(""),
    ],
    [
        "No latest election layer.",
        "No usable valid vote data.",
        "No usable margin data.",
        "Boundary caveat.",
        "County election caveat.",
    ],
    default="No major caveat."
)

display(
    model_input[[
        "target_model_ready",
        "has_latest_election_layer",
        "has_valid_vote_data",
        "has_margin_data",
        "data_confidence_note",
    ]].value_counts(dropna=False).reset_index(name="rows")
)

,target_model_ready,has_latest_election_layer,has_valid_vote_data,has_margin_data,data_confidence_note,rows
0,True,True,True,True,No major caveat.,7264
1,True,True,True,True,County election caveat.,131
2,False,True,False,False,No usable valid vote data.,98
3,False,False,False,False,No latest election layer.,57
4,True,True,True,True,Boundary caveat.,22


## 17.7 Save model input files

The national file is useful for later expansion. The North West file remains the default working scope.

In [10]:
national_path = MODEL_INPUT_DIR / "target_model_input_ward25_v1.csv"
nw_path = MODEL_INPUT_DIR / "target_model_input_north_west_ward25_v1.csv"

model_input.to_csv(national_path, index=False)

north_west_input = model_input[model_input["scope_north_west"]].copy()
north_west_input.to_csv(nw_path, index=False)

print("Saved national model input:", national_path)
print("Saved North West model input:", nw_path)
print("National rows:", len(model_input))
print("North West rows:", len(north_west_input))

# Also create per-region files if enabled and region lookup exists.
if GENERATE_REGION_OUTPUTS and "analysis_region" in model_input.columns:
    region_dir = MODEL_INPUT_DIR / "by_region"
    region_dir.mkdir(parents=True, exist_ok=True)

    for region_name, region_df in model_input.groupby("analysis_region", dropna=False):
        safe_region = re.sub(r"[^a-zA-Z0-9]+", "_", str(region_name).strip().lower()).strip("_")
        if not safe_region:
            safe_region = "unknown_region"
        out_path = region_dir / f"target_model_input_{safe_region}_ward25_v1.csv"
        region_df.to_csv(out_path, index=False)

    print("Region outputs saved:", region_dir)

Saved national model input: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs\target_model_input_ward25_v1.csv
Saved North West model input: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs\target_model_input_north_west_ward25_v1.csv
National rows: 7572
North West rows: 825
Region outputs saved: c:\Users\keena\Documents\Electoral_Tribes\data\processed\target_model_v1\inputs\by_region


## 17.8 Final readiness summary

Use this before moving to Notebook 18.

In [11]:
summary = (
    model_input
    .groupby(["analysis_region"], dropna=False)
    .agg(
        wards=("WD25CD", "nunique"),
        ready_wards=("target_model_ready", "sum"),
        with_latest_election=("has_latest_election_layer", "sum"),
        no_major_caveat=("data_confidence_note", lambda s: (s == "No major caveat.").sum()),
    )
    .reset_index()
)

summary["ready_share"] = summary["ready_wards"] / summary["wards"]

summary = summary.sort_values(["analysis_region"])
summary.to_csv(MODEL_REVIEW_DIR / "target_model_input_readiness_by_region_v1.csv", index=False)

display(summary)

,analysis_region,wards,ready_wards,with_latest_election,no_major_caveat,ready_share
0,East Midlands,761,746,761,746,0.980289
1,East of England,947,938,947,938,0.990496
2,London,689,677,677,677,0.982583
3,North East,334,334,334,334,1.000000
4,North West,825,824,825,671,0.998788
5,South East,1269,1229,1230,1229,0.968479
6,South West,821,817,817,817,0.995128
7,Wales,762,690,761,690,0.905512
8,West Midlands,754,753,754,753,0.998674
9,Yorkshire and The Humber,410,409,409,409,0.997561
